# RAG Evaluation: measure retrieval AND generation, the RAGAS way

You built a RAG pipeline — but is it **good**? "Seems fine" is not a number, and a one-line
change to the chunker or the prompt can silently regress it. RAG has **two** failure surfaces,
and you must measure each independently:

- **Retrieval** — did we fetch the right context? (*context precision*, *context recall*).
- **Generation** — is the answer **faithful** to that context (no hallucination) and **relevant**
  (does it answer the question)?

This notebook builds the RAGAS-family metrics from primitives and shows the headline contrast: a
**faithful** answer scores high faithfulness; a fluent-but-**unfaithful** one (it adds a claim not
in the context) scores measurably lower — even though both read fluently.

> **Honesty (say it up front).** ALL **retrieval metrics** (precision/recall/MRR/nDCG on real
> ranked retrieval), the **support** signal (encoder cosine of each claim vs the context), and
> **relevance** (embedding cosine) are real and measured. The illustrative pieces are **claim
> decomposition** (splitting an answer into claims) and the **judge** (deciding support/relevance)
> — in production an LLM does both; this env is encoder-only, so claim-splitting is a rule-based
> sentence splitter and the judge is the cosine threshold. Flagged again at each step.

> **Step 1 — import the canonical code + print the reproducibility banner.** Everything comes
> from `rag_evaluation.py` next to this notebook — the *same* module the page and figures use,
> which reuses chapter 5's real `DenseRetriever` and chapter 6's ranking metrics. The banner
> prints the numpy/torch **versions** and the detected **accelerator** (the encoder is CPU-pinned
> for determinism), then the corpus size, encoder backend, and support threshold.

In [1]:
import numpy as np
import torch

from rag_evaluation import (
    FAITHFUL_ANSWER,
    IRRELEVANT_ANSWER,
    QUESTION,
    SUPPORT_THRESHOLD,
    UNFAITHFUL_ANSWER,
    DenseRetriever,
    EvalSample,
    answer_relevance,
    claim_support,
    context_precision_at_k,
    context_recall,
    context_relevance,
    evaluate_sample,
    faithfulness,
    full_corpus,
)

accelerator = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print('numpy:', np.__version__)
print('torch:', torch.__version__, '| accelerator available:', accelerator, '| encoder runs on: cpu')

corpus = full_corpus()
dense = DenseRetriever(corpus)
print('corpus passages   :', len(corpus))
print('dense lens        :', dense.backend)
print('support threshold :', SUPPORT_THRESHOLD)

numpy: 2.4.6
torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus passages   : 11
dense lens        : sentence-transformers/all-MiniLM-L6-v2
support threshold : 0.5


> **Step 2 — the question and the two answers we will grade.** Both answers read fluently and
> share two **true** claims; the unfaithful one appends a hallucinated claim (a fact absent from
> the corpus). Faithfulness is the metric designed to catch exactly that difference.

In [2]:
print('QUESTION:', QUESTION)
print()
print('FAITHFUL_ANSWER  :', FAITHFUL_ANSWER)
print()
print('UNFAITHFUL_ANSWER:', UNFAITHFUL_ANSWER)

QUESTION: What is the ground resolution of the Helios-7 imager, and when was it launched?

FAITHFUL_ANSWER  : The Helios-7 imager has a ground resolution of 4 meters. Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.

UNFAITHFUL_ANSWER: The Helios-7 imager has a ground resolution of 4 meters. Helios-7 launched on March 3rd, 2024 from the Kourou spaceport. Helios-7 is powered entirely by solar panels.


## 1) Faithfulness — decompose, check each claim, aggregate

Faithfulness = (# supported claims) / (# claims). Split the answer into claims, score each
claim's support against the retrieved context (real cosine), count how many clear the threshold,
and divide. A hallucinated claim is unsupported, so it drags the fraction down.

> **Step 3 — retrieve the context both answers are graded against.** We embed the question and
> take the top-3 chunks — the real retrieval. This is the context faithfulness checks each claim
> against.

In [3]:
ranked = dense.search(QUESTION, k=len(corpus)).indices
context_chunks = tuple(corpus[i] for i in ranked[:3])
context_text = ' '.join(context_chunks)
print('retrieved context (top-3):')
for i in ranked[:3]:
    print('   doc[%d]: %s' % (i, corpus[i]))

retrieved context (top-3):
   doc[1]: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
   doc[0]: The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
   doc[10]: The Helios-7 ground team spent the afternoon investigating several telemetry errors and console warnings.


> **Step 4 — score both answers and read the per-claim breakdown.** The faithful answer's claims
> are all supported → 1.000. The unfaithful answer's hallucinated 'solar panels' claim scores
> below the threshold → unsupported → faithfulness drops to 2/3. The claim-split is the
> illustrative stand-in; the per-claim support cosines are **real**.

In [4]:
def show(result, name):
    print(f'{name}: faithfulness = {result.score:.3f} ({sum(result.supported)}/{len(result.claims)} supported)')
    for claim, support, ok in zip(result.claims, result.supports, result.supported):
        verdict = 'SUPPORTED' if ok else 'UNSUPPORTED'
        print(f'    [{verdict:<11} {support:.3f}] {claim}')

faithful = faithfulness(dense, FAITHFUL_ANSWER, context_text)
unfaithful = faithfulness(dense, UNFAITHFUL_ANSWER, context_text)
show(faithful, 'FAITHFUL  ')
print()
show(unfaithful, 'UNFAITHFUL')

FAITHFUL  : faithfulness = 1.000 (2/2 supported)
    [SUPPORTED   0.848] The Helios-7 imager has a ground resolution of 4 meters.
    [SUPPORTED   0.933] Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.

UNFAITHFUL: faithfulness = 0.667 (2/3 supported)
    [SUPPORTED   0.848] The Helios-7 imager has a ground resolution of 4 meters.
    [SUPPORTED   0.933] Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.
    [UNSUPPORTED 0.487] Helios-7 is powered entirely by solar panels.


> **Step 5 — assert the contrast before believing it.** The faithful answer is fully grounded;
> the hallucinated claim drags the unfaithful answer below it. Correctness *before* the claim.

In [5]:
assert faithful.score == 1.0, 'every claim of the faithful answer must be supported'
assert unfaithful.score < faithful.score, 'the hallucinated claim must lower faithfulness'
assert not unfaithful.supported[-1], "the 'solar panels' claim is NOT in context -> unsupported"
print(f'both answers read fluently, but faithfulness catches the hallucination: {faithful.score:.2f} vs {unfaithful.score:.2f}. All asserts passed.')

both answers read fluently, but faithfulness catches the hallucination: 1.00 vs 0.67. All asserts passed.


## 2) Answer relevance — a separate axis from faithfulness

A faithful answer can still be **useless** if it answers the wrong question. Answer relevance
measures whether the answer addresses the question (here, question↔answer embedding cosine — an
illustrative simplification of RAGAS's generate-questions-from-the-answer approach).

> **Step 6 — a grounded answer to the WRONG question scores low relevance.** The off-topic answer
> (about the project lead) is *fully faithful* — every claim is a real corpus fact — yet its
> answer relevance is low because it doesn't address the imager-resolution question. That
> dissociation is why relevance is a **separate** axis.

In [6]:
rel_on_topic = answer_relevance(dense, QUESTION, FAITHFUL_ANSWER)
rel_off_topic = answer_relevance(dense, QUESTION, IRRELEVANT_ANSWER)
off_topic_faith = faithfulness(dense, IRRELEVANT_ANSWER, ' '.join(corpus)).score
print('on-topic answer   : answer relevance = %.3f' % rel_on_topic)
print('off-topic answer  : faithfulness = %.3f (grounded), answer relevance = %.3f (off-topic)'
      % (off_topic_faith, rel_off_topic))
assert off_topic_faith == 1.0, 'the off-topic answer is still fully grounded'
assert rel_on_topic > rel_off_topic, 'the on-topic answer must be more relevant'
print(f'\n-> faithful ({off_topic_faith:.2f}) but irrelevant ({rel_off_topic:.3f}): relevance is a SEPARATE axis.')

on-topic answer   : answer relevance = 0.906


off-topic answer  : faithfulness = 1.000 (grounded), answer relevance = 0.458 (off-topic)

-> faithful (1.00) but irrelevant (0.458): relevance is a SEPARATE axis.


## 3) Retrieval metrics — context precision@k and recall

The other failure surface: did we even retrieve the right context? **Context precision@k** rewards
ranking relevant chunks high; **context recall** measures whether we retrieved them at all. These
localize a failure to *retrieval*, upstream of generation.

> **Step 7 — a good ranking vs a buried one.** The relevant chunks are the imager (1) and launch
> (0) passages. The real retrieval ranks them up top (precision/recall 1.0); a deliberately bad
> ranking buries them below the top-3 (precision/recall 0.0). Same relevant set, opposite scores —
> the metric points straight at the retriever.

In [7]:
relevant = frozenset({0, 1})  # imager + launch chunks
k = 3
good = dense.search(QUESTION, k=len(corpus)).indices
distractors = tuple(i for i in range(len(corpus)) if i not in relevant)
bad = distractors[:3] + tuple(sorted(relevant)) + distractors[3:]
print('GOOD ranking top-%d: %s' % (k, list(good[:k])))
print('   context precision@%d = %.3f' % (k, context_precision_at_k(good, relevant, k)))
print('   context recall@%d    = %.3f' % (k, context_recall(good, relevant, k)))
print('BAD  ranking top-%d: %s' % (k, list(bad[:k])))
print('   context precision@%d = %.3f' % (k, context_precision_at_k(bad, relevant, k)))
print('   context recall@%d    = %.3f' % (k, context_recall(bad, relevant, k)))
assert context_precision_at_k(good, relevant, k) > context_precision_at_k(bad, relevant, k)
assert context_recall(bad, relevant, k) == 0.0, 'the bad ranking retrieves NO relevant chunk in top-3'
print('\n-> precision/recall localize the failure to RETRIEVAL.')

GOOD ranking top-3: [1, 0, 10]
   context precision@3 = 1.000
   context recall@3    = 1.000
BAD  ranking top-3: [2, 3, 4]
   context precision@3 = 0.000
   context recall@3    = 0.000

-> precision/recall localize the failure to RETRIEVAL.


## 4) The RAGAS / TruLens triad

The three generation-side lenses together: **context relevance** (is the context on-topic?),
**faithfulness** (is the answer grounded?), **answer relevance** (does it answer the question?).
All three high = a trustworthy answer; any one low points at a different fix.

> **Step 8 — measure all three legs on the demo.** Each is a real number. A low leg is diagnostic:
> low context relevance → retrieval; low faithfulness → the generator hallucinated; low answer
> relevance → it answered the wrong thing.

In [8]:
cr = context_relevance(dense, QUESTION, context_chunks)
fa = faithfulness(dense, FAITHFUL_ANSWER, context_text).score
ar = answer_relevance(dense, QUESTION, FAITHFUL_ANSWER)
print('context relevance : %.3f  (is the retrieved context on-topic?)' % cr)
print('faithfulness      : %.3f  (is every claim grounded?)' % fa)
print('answer relevance  : %.3f  (does it answer the question?)' % ar)
assert cr > 0.5 and fa == 1.0 and ar > 0.5, 'the good answer should score high on all three legs'
print('\n-> all three legs high -> a trustworthy answer, by three independent lenses.')

context relevance : 0.603  (is the retrieved context on-topic?)
faithfulness      : 1.000  (is every claim grounded?)
answer relevance  : 0.906  (does it answer the question?)

-> all three legs high -> a trustworthy answer, by three independent lenses.


## 5) The pitfall: cosine support ≈ topic, not entailment

The honest limit carried forward from chapter 8: the cosine support proxy measures **topical**
similarity, not factual **entailment**. A hallucinated *number* is topically almost identical to
the truth, so it slips past a raw-cosine gate.

> **Step 9 — a number-swap slips past the cosine bar.** Swap '4 meters' → '2 meters': the false
> claim is topically near-identical to the true one, so its support cosine still **clears** the
> threshold. It scores below the true claim, but a raw-cosine gate can't catch a plausible
> fact-swap — which is exactly why RAGAS uses an LLM judge (an NLI-style entailment decision).

In [9]:
imager_ctx = corpus[1]  # 'ground resolution of 4 meters'
true_claim = 'The Helios-7 imager has a ground resolution of 4 meters.'
swap_claim = 'The Helios-7 imager has a ground resolution of 2 meters.'  # WRONG number
s_true = claim_support(dense, true_claim, imager_ctx)
s_swap = claim_support(dense, swap_claim, imager_ctx)
print('context   :', imager_ctx)
print('TRUE  (4m): support %.3f  (>= %.1f: %s)' % (s_true, SUPPORT_THRESHOLD, s_true >= SUPPORT_THRESHOLD))
print('SWAP  (2m): support %.3f  (>= %.1f: %s)' % (s_swap, SUPPORT_THRESHOLD, s_swap >= SUPPORT_THRESHOLD))
assert s_swap >= SUPPORT_THRESHOLD, 'the number-swap SLIPS PAST the cosine bar (a real limit)'
assert s_true > s_swap, 'the true claim still scores higher'
print('\n-> the false 2m claim clears the bar: cosine ~ topical, not factual. Why RAGAS uses an LLM judge.')

context   : Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
TRUE  (4m): support 0.848  (>= 0.5: True)
SWAP  (2m): support 0.836  (>= 0.5: True)

-> the false 2m claim clears the bar: cosine ~ topical, not factual. Why RAGAS uses an LLM judge.


## 6) One golden record → all metrics

A real eval set is a list of **golden records** — each a `(question, answer, relevant-set)` triple.
The harness `evaluate_sample` runs *every* metric for one such record and returns an `EvalReport`.
This is the unit you'd loop over an eval set with, and gate in CI.

> **Step 10 — one record → a full `EvalReport`, cross-checked.** We build one golden `EvalSample`
> (the imager+launch question, the faithful answer, relevant chunks {0, 1}) and run the harness. It
> retrieves, then computes retrieval metrics (precision/recall/MRR/nDCG) and generation metrics
> (context relevance, faithfulness, answer relevance) in one call. The asserts confirm the report's
> fields **exactly match** the numbers we computed one-by-one in Steps 4–8 — so the harness is just
> those same metrics, packaged per record. (MRR 0.500 and nDCG@3 0.631 are new: they use the first
> relevant chunk as a single-gold anchor, which sits at rank 2 → 1/2 and 1/log₂3.)

In [10]:
sample = EvalSample(
    question=QUESTION,
    answer=FAITHFUL_ANSWER,
    relevant=frozenset({0, 1}),  # imager + launch chunks
    label='imager+launch, faithful answer',
)
report = evaluate_sample(dense, corpus, sample, k=3)
print('sample:', sample.label)
print('  retrieved top-3     :', list(report.retrieved))
print('  context precision@3 : %.3f' % report.context_precision)
print('  context recall@3    : %.3f' % report.context_recall)
print('  MRR (first-gold)    : %.3f' % report.mrr)
print('  nDCG@3 (first-gold) : %.3f' % report.ndcg)
print('  context relevance   : %.3f' % report.context_relevance)
print('  faithfulness        : %.3f' % report.faithfulness.score)
print('  answer relevance    : %.3f' % report.answer_relevance)

# the report's fields must agree with the numbers computed one-by-one in Steps 4-8
assert report.faithfulness.score == faithful.score, 'report faithfulness must match Step 4'
assert report.answer_relevance == rel_on_topic, 'report answer relevance must match Step 6'
assert report.context_relevance == cr, 'report context relevance must match Step 8'
assert report.context_precision == 1.0 and report.context_recall == 1.0, 'both relevant chunks in top-3'
assert report.mrr == 0.5 and abs(report.ndcg - 0.631) < 1e-3, 'single-gold anchor sits at rank 2'
print('\n-> one record -> every metric, all agreeing with the per-metric calls above. This is the CI unit.')

sample: imager+launch, faithful answer
  retrieved top-3     : [1, 0, 10]
  context precision@3 : 1.000
  context recall@3    : 1.000
  MRR (first-gold)    : 0.500
  nDCG@3 (first-gold) : 0.631
  context relevance   : 0.603
  faithfulness        : 1.000
  answer relevance    : 0.906

-> one record -> every metric, all agreeing with the per-metric calls above. This is the CI unit.


## Try it yourself

Before you run the next cell, **predict**. The unfaithful answer had **3** claims, **2** supported
→ faithfulness **0.667**. Now suppose we add a **second** hallucinated claim (another fact absent
from the context) — so **4** claims, still **2** supported.

**Predict:** what will faithfulness be — higher, lower, or the same as 0.667? And will the two
original true claims still be marked supported?

Then run it and check. *(Hint: faithfulness = supported / total. Adding an unsupported claim grows
the denominator but not the numerator, so the fraction falls: 2/4 = 0.5. Faithfulness punishes
*every* ungrounded claim — more hallucination, lower score.)* The notebook asserts **0.5**.

In [11]:
two_hallucinations = (
    FAITHFUL_ANSWER  # the two true claims
    + ' Helios-7 is powered entirely by solar panels.'          # hallucination 1 (not in context)
    + ' Helios-7 carries twelve scientific instruments on board.'  # hallucination 2 (not in context)
)
result = faithfulness(dense, two_hallucinations, context_text)
print('claims:', len(result.claims), '| supported:', sum(result.supported),
      '| faithfulness = %.3f' % result.score)
for claim, ok in zip(result.claims, result.supported):
    print('   [%s] %s' % ('SUPPORTED  ' if ok else 'UNSUPPORTED', claim))
assert result.score == 0.5, 'two of four claims supported -> faithfulness 0.5'
assert sum(result.supported) == 2, 'the two original true claims stay supported'
print('\n-> 2/4 = 0.5: a second hallucination drops faithfulness further. More ungrounded claims, lower score.')

claims: 4 | supported: 2 | faithfulness = 0.500
   [SUPPORTED  ] The Helios-7 imager has a ground resolution of 4 meters.
   [SUPPORTED  ] Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.
   [UNSUPPORTED] Helios-7 is powered entirely by solar panels.
   [UNSUPPORTED] Helios-7 carries twelve scientific instruments on board.

-> 2/4 = 0.5: a second hallucination drops faithfulness further. More ungrounded claims, lower score.


## What we saw

- **Faithfulness caught a hallucination two fluent answers hid** — 1.000 (all grounded) vs 0.667
  (the 'solar panels' claim scored below the support threshold).
- **Answer relevance is a separate axis** — a fully-faithful answer to the *wrong* question scored
  low relevance (0.458 vs 0.906). Grounded ≠ useful.
- **Retrieval metrics localize failures** — context precision@3 / recall@3 were 1.0 on the real
  ranking and 0.0 when the relevant chunks were buried, pointing straight at the retriever.
- **The RAGAS triad** — context relevance + faithfulness + answer relevance, three independent
  lenses; a low leg tells you *which* stage to fix.
- **The honest limit** — the cosine support proxy measures *topic*, not *entailment*: a '2 meters'
  number-swap slipped past the bar, which is why RAGAS uses an LLM judge.
- **One golden record → all metrics** — `evaluate_sample` runs every retrieval + generation metric
  for one `(question, answer, relevant-set)` record and returns an `EvalReport` whose fields match
  the per-metric numbers exactly. That is the unit you loop over an eval set with, and gate in CI.

**Real vs illustrative:** every retrieval metric, support cosine, and relevance sim is **real and
measured**; only **claim decomposition** and the **judge** are illustrative stand-ins for an LLM
(a rule-based splitter + a cosine threshold), so the whole pipeline runs reproducibly with no
generative model.